In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib
import re
import os
import matplotlib.cm as cm
from collections import Counter
import glob
import shutil
from problemsolver.generator import build_pareto_for_dir

%load_ext autoreload
%autoreload 2

%matplotlib inline

RESULT_ROOT = pathlib.Path(os.getenv("RESULT_ROOT")).expanduser().resolve()  # Set this environment variable to the location in which all your results are stored

def add_names_to_df(df):
    NAMING_PATTERN = re.compile(r"^(.*?)(?:_(\d+))?$")  # This splits a string like "minimize_BirdFlockingAlgorithm_5" on the final underscore, producing "minimize_BirdFlockingAlgorithm" and "5"
    df[['base', 'suffix']] = df['method_name'].str.extract(NAMING_PATTERN)
    df['base'] = df['base'].str.replace('minimize_', '')
    return df

# Setup

Copy the baseline problem template, including optimizers_performant.csv which needs to be 

Transfer all data to local directory:
`rsync -avz eric@tuxbox:~/research/problemsolver/ ~/Documents/Research/ProblemSolver/problemsolver/`

Running a blend2 run:
```
python ~/src/problemsolver/src/problemsolver/generator.py blend-sweep \
--n-blend-examples 2 \
--api-key <API_KEY> \
--api-base https://api.novita.ai/openai \
--model deepseek/deepseek-v3.1 \
--n-tuning-trials 50 \
--n-tune-functions 50 \
--n-test-functions 20 \
--n-dims 2 \
--output-dir ~/research/problemsolver/blend2_deepseek31 \
--pareto-rtol 0.3 \
--n-jobs 4
```

# Result preparation

## File manipulation and experiment management: Consolidate results from many different experiment folders

Required steps:
1. Copy result files like "optimizers_{ftype}.csv" from the individual experiment folder into a `consolidated_results/` folder
2. Using patterns in each folder name, create a mapping of experiments into experiment types, and experiments into stems (allowing comparison within an experiment group)
3. 

In [ ]:
# STEP 1: CONSOLIDATE EXPERIMENTS AND PERFORMANT RESULTS: OVERWRITE THE PERFORMANT RESULTS IF DESIRED
experiments_dir = RESULT_ROOT / "problemsolver"
consolidated_dir = RESULT_ROOT / "consolidated_results"
plot_dir = RESULT_ROOT / "plots"

# Build a list of experiments, i.e. the top-level directories in the experiments_dir
candidate_paths = glob.glob(str(experiments_dir) + '/*/')
experiments = []
for p in sorted(candidate_paths):
    ps = pathlib.Path(p).stem
    if 'problemsolver' not in ps and 'archive' not in ps:
        experiments.append(ps)

# Copy the results
for exp in experiments:
    for ftype in ['all', 'performant']:
        src = experiments_dir / exp / f"optimizers_{ftype}.csv"
        dest_dir = consolidated_dir / ftype
        dest_dir.mkdir(parents=True, exist_ok=True)
        dst = dest_dir / f"{exp}.csv"
        if src.exists():
            shutil.copy(src, dst)
            print(f"✅ Copied {src.parent.stem}/{src.name} → {dst.parent.stem}/{dst.name}")
        else:
            print(f"⚠️ Skipped {folder}: source file not found ({src})")

# Step 2: 
experiment_type_prefixes = {'blend2_': 'blend2'}
experiment_stem = {}
experiment_type_map = {}
for e in experiments:
    matching_experiment_types = []
    for lbl in experiment_type_prefixes:
        if lbl in e:
            matching_experiment_types.append(lbl)
    if len(matching_experiment_types) == 0:
        experiment_type_map[e] = 'base'
        experiment_stem[e] = e
    elif len(matching_experiment_types) == 1:
        experiment_type_map[e] = experiment_type_prefixes[matching_experiment_types[0]]
        experiment_stem[e] = e.replace(matching_experiment_types[0], '')
    else:
        raise AssertionError(f"Experiment name clash: {matching_experiment_types} for {e}")    

basic_experiments = [e for e in experiments if 'blend2' not in e]
blend2_experiments = [e for e in experiments if 'blend2' in e]

print(f"Basic Experiments: {basic_experiments}")
print(f"Blend2 Experiments: {blend2_experiments}")


# Step 3: Load the priors (optimizers used to construct the initial pareto frontier) for use in plotting and reference
# Pull out a dataframe of just the reference priors which were provided as the basis of the initial pareto frontier against which subsequent models are evaluated
reference_priors = pd.read_csv(consolidated_dir / "optimizers_performant.csv")
# reference_priors = pd.read_csv(RESULT_ROOT / 'baseline_models_results' / 'updated_baseline_pareto.csv')
reference_priors = add_names_to_df(reference_priors)
reference_priors.loc[:, 'loss'] = reference_priors.loc[:, 'log_rel_error'] + reference_priors.loc[:, 'time_elapsed'] * 100

In [ ]:
# OPTIONAL: Re-compute the pareto frontier for each experiment based on a custom frontier configuration
# NOTE: This will overwrite the existing optimizers_performant.csv file!
# This can be used to remove path dependence in the testing of pareto optimality (recompute_pareto_frontier=False)
build_pareto_for_dir(input_dir=str(consolidated_dir / "all"),
                     benchmark_fname=str(consolidated_dir / "optimizers_performant.csv"),
                     # benchmark_fname=str(RESULT_ROOT / 'baseline_models_results' / 'updated_baseline_pareto.csv'),
                     output_dir=str(consolidated_dir / "performant"),
                     pareto_metric='strict',
                     rtol=0.0,
                     recompute_pareto_frontier=False,
                     pattern="*.csv",
                     log_level='warning',
                     )

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))

In [ ]:
def ensure_df_has_priors(df, priors) -> pd.DataFrame:
    # If the dataframe doesn't have the priors, add them
    if not df['suffix'].isnull().any():
        df = pd.concat([priors, df])
    return df

def plot_from_experiment_table(df, ax=None):
    # 3. Determine unique base methods WITHOUT suffix
    base_methods = sorted(df[df['suffix'].isna()]['base'].unique())
    novel_methods = sorted(df[df['suffix'].notnull()]['base'].unique())
    
    # 4. Build a color map for those base methods
    cmap_name = 'tab20' if len(novel_methods) > 10 else 'tab10'
    cmap = plt.get_cmap(cmap_name, len(novel_methods))
    novel_method_colors = {name: cmap(i) for i, name in enumerate(novel_methods)}
    
    # 5. Now plot
    if not ax:
        fig, ax = plt.subplots(figsize=(8,6))

    plotted_bases = []
    
    for i, (_, row) in enumerate(df.iterrows()):
        x = row['time_elapsed']
        y = row['log_rel_error']
        base = row['base']
        is_prior = pd.isna(row['suffix'])
        zorder = i + is_prior * len(df)

        if row['base'] == 'constant':
            continue  # Skip the artificial constant entry used for defining the pareto frontier
        
        color = 'black' if is_prior else novel_method_colors[base]
        ax.scatter(x, y, color=color, s=50, edgecolors='w', linewidth=0.5, zorder=zorder)
        if row['base'] not in plotted_bases:
            ax.text(x+0.0005, y, row['base'],
                    fontsize=8, va='center', ha='left', 
                    color=color, zorder=zorder)
            plotted_bases.append(row['base'])
    
    # 6. Polish axes
    xmin, xmax = ax.get_xlim()
    if xmax > 0.03:
        ax.set_xscale('log')
    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin, 1)
    ax.set_xlabel('Time Elapsed (s)')
    ax.set_ylabel('Log Relative Error')
    
    
    ax.grid(True, linestyle='--', alpha=0.5)
    return ax

# Plot results in performance space

experiments = experiments
# experiments = basic_experiments
# experiments = blend2_experiments

for experiment_name in experiments:
    for optimizer_type in ['performant', 'all']:
        print(f"{experiment_name} with {optimizer_type}")
        df = pd.read_csv(consolidated_dir / optimizer_type / f"{experiment_name}.csv")
        df = add_names_to_df(df)
        df = ensure_df_has_priors(df, reference_priors)
        
        print(f"{experiment_name}: {df.shape}")
        plot_df = df[df['log_rel_error'] <= 1.0].copy()
        print(f"Plotting {len(plot_df)} of {len(df)} experiments")
        ax = plot_from_experiment_table(plot_df)
        ax.set_title(f'Comparison of {optimizer_type} Optimization Methods: {experiment_name}')
        plt.tight_layout()
        plt.savefig(plot_dir / f"{experiment_name}_{optimizer_type}.png")
        plt.show()

In [ ]:
# Side-by-side plot for header
experiment_name = 'gpt_oss_120b'
optimizer_type='performant'
df = pd.read_csv(consolidated_dir / optimizer_type / f"{experiment_name}.csv")
df = add_names_to_df(df)
df = ensure_df_has_priors(df, reference_priors)
plot_df = df[df['log_rel_error'] < 1.0].copy()

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(12,5))
only_priors = plot_df[plot_df['best_params'].isnull()]
full_ax = plot_from_experiment_table(plot_df, ax=ax[1])
prior_ax = plot_from_experiment_table(only_priors, ax=ax[0])
prior_ax.set_xlim(*full_ax.get_xlim())
prior_ax.set_xscale('log')
prior_ax.set_ylim(*full_ax.get_ylim())
prior_ax.set_title('Nonconvex Optimizer Performance')
full_ax.set_title('Nonconvex Optimizer Performance...With LLMS')
plt.tight_layout()
plt.savefig(plot_dir / f"{experiment_name}_performance_sidebyside.png")

In [ ]:
"""
Consolidate experiments and their priors, with additional data fields:
- experiment_number: Ordered position in the results table, *excluding* priors
- index: Ordered position in the results table, *including* priors
"""

def build_experiment_data_from_files(experiments, input_dir, result_type='all'):
    all_experiment_data = []
    all_prior_data = {}    
    for experiment_name in experiments:
        df = pd.read_csv(input_dir / f"{experiment_name}.csv")
        df = add_names_to_df(df)
        df.loc[:, 'result_type'] = result_type
        df.loc[:, 'model'] = experiment_stem[experiment_name]
        df.loc[:, 'experiment_type'] = experiment_type_map[experiment_name]
        df.loc[:, 'experiment_name'] = experiment_name
        non_prior_data = df[df['suffix'].notnull()].reset_index()
        non_prior_data.index.name = 'experiment_number'  # re-index from 0, but keep the original numbering as the "experiment_number" column
        non_prior_data = non_prior_data.reset_index()
        all_experiment_data.append(non_prior_data)
        all_prior_data[experiment_name] = df[df['suffix'].isnull()]

    df = pd.concat(all_experiment_data)
    df = df.reset_index().set_index(['model', 'experiment_type', 'result_type', 'experiment_name', 'experiment_number', 'base', 'suffix'])
    # df = df.sort_index()
    df = df.reset_index('experiment_number')
    df['loss'] = df['log_rel_error'] + 100 * df['time_elapsed']
    return df
    
res = []
for result_type in ['all', 'performant']:
    res.append(build_experiment_data_from_files(experiments, consolidated_dir / result_type, result_type=result_type))

all_results_df = pd.concat(res)

# df = build_experiment_data_from_files(experiments, consolidated_dir)
# df_without_priors = df.drop('0_priors', level='experiment_name')
loss_series = all_results_df['loss'].dropna()
df.head(3)

# df = df[df['loss'] <= 1.0]  # There can be some high-loss solutions which are outside of the normal bounds (e.g. high inaccuracy, low speed) which should probably be barred in the original experiment pipe

# We have 
# All experiments should now be zero-indexed by experiment number, 
#   i.e. the position in which they were added to the result list (not counting the priors) 

In [ ]:
experiment_count = loss_series.groupby(level=['model', 'experiment_type', 'result_type']).count()
result_stats = pd.DataFrame.from_dict({'count all': experiment_count.xs('all', level='result_type'),
                         'pass_rate': experiment_count.xs('performant', level='result_type') / experiment_count.xs('all', level='result_type').rename('pass_rate')
                         })
display(result_stats)

In [ ]:
experiment_types = result_stats.index.get_level_values('experiment_type').unique()

for exp in experiment_types:
    print(exp)
    subset_df = result_stats.xs(exp, level='experiment_type')
    subset_df.to_csv(plot_dir / f"summary_{exp}.csv")
    display(subset_df)

## How do base and "Blend2" approaches differ?

In [ ]:
plot_series = loss_series[loss_series <= 1.1]
plot_series = plot_series.xs('performant', level='result_type')
print(plot_series.groupby('experiment_name').count())
plot_df = plot_series.reset_index()
ax = sns.violinplot(plot_series.reset_index(), x='model', y='loss', hue='experiment_type', split=True)
ax.tick_params(axis='x', rotation=90)
plt.tight_layout()
plt.savefig(plot_dir / "experiment_violinplot.png")
plt.show()

In [ ]:
experiment_types = loss_series.index.get_level_values('experiment_type').unique()
result_types = loss_series.index.get_level_values('result_type').unique()

fig, ax = plt.subplots(nrows=len(experiment_types), ncols=len(result_types), sharex='col')

for r, experiment_type in enumerate(experiment_types):
    for c, result_type in enumerate(result_types):
        cax = ax[r][c]
        subset = loss_series.xs((experiment_type, result_type), level=('experiment_type', 'result_type'))
        for m in subset.index.get_level_values('model').unique():
            cax.ecdf(subset[m], label=m)
        if c == 0:
            cax.set_ylabel(f'CDF for {experiment_type} test')
            if experiment_type == "base":
                cax.legend()
        if r == 0:
            cax.set_title(result_type)
        if r == len(experiment_types) - 1:
            cax.set_xlabel('Combined Loss')

plt.tight_layout()
plt.savefig(plot_dir / "experiment_cdf_subplots.png")
plt.show()

### Statistical test of differences in performant models

In [ ]:
from scipy.stats import ttest_ind

def ttest_and_print(test, ref, lbl):
    if test.mean() > ref.mean():
        print(f"{m} test mean is greater than reference mean; skipping")
        return None
    else:
        ttest_res = ttest_ind(a=ref,
                              b=test
                             )
        print(f"{lbl} 1-sided t-test: {ttest_res.pvalue / 2}")
        return ttest_res

plot_series = loss_series[loss_series <= 1.1]
plot_series = plot_series.xs('performant', level='result_type')
test_models = ['deepseek_v31', 'gpt_4_1_mini', 'gpt_oss_120b']
plot_series = plot_series[['deepseek_v31', 'gpt_4_1_mini', 'gpt_oss_120b']] # Only models which are shared between scenarios
    
for m in test_models:
    ttest_and_print(ref=plot_series[m]['base'],
                    test=plot_series[m]['blend2'],
                    lbl=m
                   )
_ = ttest_and_print(ref=plot_series.xs('base', level='experiment_type'),
                test=plot_series.xs('blend2', level='experiment_type'),
                lbl='All data'
               )
    

# Are results getting better between attempts?


In [ ]:
multi_attempt_series = all_results_df.xs('all', level='result_type')['loss']
multi_attempt_series = multi_attempt_series[~multi_attempt_series.index.duplicated(keep='last')]
loss_improvement = multi_attempt_series - multi_attempt_series.groupby(level=['experiment_name', 'base']).transform("last")
suffix_series = pd.Series(loss_improvement.index.get_level_values(level='suffix'), index=loss_improvement.index).astype(int)
suffix_offset = suffix_series - suffix_series.groupby(level=['experiment_name', 'base']).transform("last")
offset_df = pd.DataFrame.from_dict({'loss': loss_improvement, 
                                    'suffix_offset': suffix_offset}, orient='columns')
offset_df = offset_df.set_index('suffix_offset', append=True).squeeze()
offset_results = offset_df.reset_index(level='suffix', drop=True).unstack(level='suffix_offset')
offset_results_noindex = offset_results.reset_index(drop=True).T

plot_df = offset_results.reset_index(drop=True).T
ax = plot_df.plot(alpha=0.05, color='tab:blue', legend=False)
plot_df.median(1).plot(linestyle='--', color='tab:green', marker='o', label='median')
plot_df.mean(1).plot(linestyle='--', color='tab:red', marker='o', label='mean')
plt.axhline(0, color='k', linestyle='--')
ax.set_ylim(-2, 2)
ax.set_xlim(-4, -1)

ax.set_xticks([-4, -3, -2, -1])
ax.set_xticklabels(['4', '3', '2', '1'])

handles, labels = ax.get_legend_handles_labels()
keep = [i for i, lbl in enumerate(labels) if lbl in ['median', 'mean']]
ax.legend([handles[i] for i in keep], [labels[i] for i in keep])
plt.ylabel("loss(current code version) - loss(final code version)")
plt.xlabel("Number of revisions before final code version")
plt.title("Improvement in results over multiple optimizer code versions")

plt.tight_layout()
plt.savefig(plot_dir / "improvement_over_experiment_attempts.png")

# Are results getting better through evolution?

For the "Blend 2 references" approach, we expect result quality to improve as our reference library of models becomes more performant.

For this, we need to re-create the function pool over time, which required re-computing the pareto frontier to identify new additions.

In [ ]:
# Recreate the library of algorithms which were available to the model at each optimizer generation

build_pareto_for_dir(input_dir=str(consolidated_dir / "all"),
                     benchmark_fname=str(consolidated_dir / "optimizers_performant.csv"),
                     output_dir=str(consolidated_dir / "performant_recalculating"),
                     pareto_metric='strict',
                     rtol=0.3,
                     recompute_pareto_frontier=True,
                     pattern="*.csv",
                     log_level='warning',
                     )

continually_resetting_df = build_experiment_data_from_files(experiments, consolidated_dir / "performant_recalculating", result_type='performant')
loss_series = continually_resetting_df['loss'].dropna()
plot_series = loss_series.xs(('performant', 'blend2'), level=['result_type','experiment_type'])
models = plot_series.index.get_level_values('model').unique()

tmp_reference_priors = reference_priors.rename({"method_name":"base"}, axis=1)
ref_prior_series = tmp_reference_priors.set_index('base')['loss']

for m in models:
    subset = plot_series[m]
    subset.reset_index(drop=True).plot(title=f"Evolution of algorithm pool performance for {m}")

    plot_series_with_priors = pd.concat([ref_prior_series.reset_index(), subset.reset_index()]).set_index(subset.index.names).squeeze()
    expanding_mean = plot_series_with_priors.expanding().mean().loc[subset.index]
    expanding_mean.reset_index(drop=True).plot(linestyle='--', color='k', alpha=0.5)
    plt.axhline(ref_prior_series.mean(), linestyle='--', color='g', alpha=0.5)
    xticks = plt.xticks()[0]
    plt.xticks(ticks=xticks.astype(int))
    plt.xlim(xticks[1], xticks[-2])
    plt.ylabel('Total loss')
    plt.xlabel('Algorithm instance')
    plt.tight_layout()
    plt.savefig(plot_dir / f"evolution_{m}.png")
    plt.show(); plt.close()

# Additional operations

In [ ]:
# Re-creating baseline based on a new run with inspiration prompts drawn from the literature

experiment_stem['baseline_models'] = 'baseline'
experiment_type_map['baseline_models'] = 'base'

ref_res = {}
for result_type in ['all', 'performant']:
    ref_res[result_type] = build_experiment_data_from_files(['baseline_models'], RESULT_ROOT / 'baseline_models_results' / result_type, result_type=result_type)

performant_res = ref_res['performant']
performant_res = performant_res[performant_res['loss'].notnull()]
performant_res = performant_res.loc[performant_res.groupby(level='base')['loss'].idxmin(),:]
all_res = ref_res['all']
all_res = all_res[all_res['loss'].notnull()]

print(f"Raw performant experiments: {performant_res.shape[0]} results")
print(f"Raw all-experiments: {all_res.shape[0]} results")

all_res_best = all_res.loc[all_res.groupby(level='base')['loss'].idxmin(),:]
res_not_in_performant = all_res_best[~all_res_best.index.get_level_values('base').isin(performant_res.index.get_level_values('base').unique())]
consolidated_performant_res = pd.concat([performant_res, res_not_in_performant])
output = consolidated_performant_res.reset_index(level='base')[['base', 'log_rel_error', 'time_elapsed']].reset_index(drop=True)
output['method_name'] = 'minimize_' + output['base']
output.drop('base', axis=1).set_index('method_name').to_csv(RESULT_ROOT / 'baseline_models_results' / 'updated_baseline_pareto.csv')
print(f"After dropping duplicates and nuls, {output.shape[0]} results")
output.set_index('base').plot.scatter(x='time_elapsed', y='log_rel_error', logx=True)

# Limitations

General limitations with the framework and analysis:
- We have not done a rigorous study on the stability of the test metrics (log relative error, computation speed) and the suitability of the hyperparameter optimization at the sample size which we are running.  
- In practice, we only consider the (simple) 2-D optimization case to reduce the computational complexity associated with convergence. We do not have a guarantee that the generated methods extend to higher dimensions, or that their performance scales to higher dimensions.
- For budget reasons, we mostly work with open-source models. The performance of o4 is clearly a huge improvement, and 

Limitations on the Blend-2 analysis:
 